# ROOTS Step 4: Receptor and Ligand Preparation

**Input:**
- ALDH3B2 validated structure from Step 2 (`ALDH3B2_AF3.pdb`)
- FOLR1 benchmark structures from PDB (4LRH folate-bound, 5IZQ antifolate-bound)
- Filtered compound library from Step 3 (`library_final.csv`, 328,869 compounds)

**What this notebook does:**
1. Downloads and validates both receptor structures (ALDH3B2 from Drive, FOLR1 from PDB)
2. Runs PDB2PQR protonation at physiological pH (7.4) with PROPKA titration
3. Converts protonated receptors to PDBQT via Meeko (`mk_prepare_receptor.py`)
4. Converts the compound library SMILES to docking-ready PDBQT via Meeko (`mk_prepare_ligand.py`)
5. Saves all prepared files to Drive for Step 5 (docking validation) and Step 6 (high-throughput docking)

**What this notebook does NOT do:** define the docking box (grid center/size), that belongs in Step 5 where it's validated against a known reference ligand before the real screen begins.

**CHARMM-GUI note:** the locked methodology names CHARMM-GUI alongside PDB2PQR for protonation. CHARMM-GUI's Solution Builder is a web-only interface with no scriptable API. For the docking-preparation context specifically (assigning protonation states and converting to PDBQT), PDB2PQR + Meeko is the correct, fully scriptable path. CHARMM-GUI becomes relevant later at Step 10 (MD system preparation), where it generates GROMACS-ready topology files, a genuinely different task from docking prep.

## 0. Environment setup and self-healing session state

In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess, shutil, os

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount('/content/drive')

DRIVE_BASE = Path("/content/drive/MyDrive/ROOTS_Pipeline")
STEP2_DIR = DRIVE_BASE / "Step2"
STEP2A_DIR = DRIVE_BASE / "Step2a"
STEP3_DIR = DRIVE_BASE / "Step3"
STEP4_DIR = DRIVE_BASE / "Step4"
STEP4_DIR.mkdir(parents=True, exist_ok=True)
(STEP4_DIR / "receptors").mkdir(exist_ok=True)
(STEP4_DIR / "ligands").mkdir(exist_ok=True)
(STEP4_DIR / "logs").mkdir(exist_ok=True)

print(f"Timestamp: {datetime.now().isoformat()}")
print(f"Step 4 output directory: {STEP4_DIR}")

In [ ]:
# Install required packages
# PDB2PQR: pip-installable, provides the pdb2pqr30 CLI and Python module
# Meeko: pip-installable, provides mk_prepare_receptor.py and mk_prepare_ligand.py
# Note: Meeko requires python >=3.9, <3.12 per its PyPI metadata — if this
# Colab runtime is 3.12+, the install may fail or behave unexpectedly; check
# the version and flag it rather than discovering it mid-run.

import sys
print(f"Python version: {sys.version}")
if sys.version_info >= (3, 12):
    print("WARNING: Meeko officially supports python <3.12. This runtime is "
          f"{sys.version_info.major}.{sys.version_info.minor}. If meeko install "
          "fails below, this is the likely cause — not a network or pip issue.")

%pip install -q pdb2pqr meeko rdkit biopython

# Verify installs
for pkg in ['pdb2pqr', 'meeko', 'rdkit']:
    try:
        __import__(pkg if pkg != 'rdkit' else 'rdkit')
        print(f"  {pkg}: installed")
    except ImportError:
        print(f"  {pkg}: FAILED — see warning above if python version is >=3.12")

## 1. Locate and validate receptor structures

Two receptors, two different sources:
- **ALDH3B2**: AF3-predicted, validated in Step 2 (Section 5b), expected on Drive
- **FOLR1**: PDB experimental structures, downloaded fresh from RCSB here

For FOLR1, two structures are relevant:
- **4LRH**: folate-bound (native ligand complex, defines the natural pocket)
- **5IZQ**: antifolate AGF183-bound (small-molecule-occupied, proves a non-folate compound can sit in this pocket, directly relevant as re-docking reference for Step 5)

In [ ]:
from Bio.PDB import PDBList, PDBParser
import requests

# ---- ALDH3B2: load from Drive (Step 2 output) ----
aldh3b2_path = STEP2_DIR / "structures" / "ALDH3B2_AF3.pdb"
if not aldh3b2_path.exists():
    # Try Step2a location as fallback
    aldh3b2_path = STEP2A_DIR / "structures" / "ALDH3B2_AF3.pdb"

if aldh3b2_path.exists():
    print(f"ALDH3B2 structure found: {aldh3b2_path}")
    shutil.copy2(aldh3b2_path, STEP4_DIR / "receptors" / "ALDH3B2_raw.pdb")
else:
    print(f"ALDH3B2 structure NOT found at either:")
    print(f"  {STEP2_DIR / 'structures' / 'ALDH3B2_AF3.pdb'}")
    print(f"  {STEP2A_DIR / 'structures' / 'ALDH3B2_AF3.pdb'}")
    print("Complete Step 2 (AF3 submission + model selection) before running this cell.")

# ---- FOLR1: download from PDB ----
pdbl = PDBList()
for pdb_id, label in [("4LRH", "folate_bound"), ("5IZQ", "antifolate_AGF183")]:
    print(f"\nDownloading FOLR1 {pdb_id} ({label})...")
    pdb_file = pdbl.retrieve_pdb_file(pdb_id, file_format='pdb',
                                        pdir=str(STEP4_DIR / "receptors"))
    if pdb_file and os.path.exists(pdb_file):
        clean_name = STEP4_DIR / "receptors" / f"FOLR1_{pdb_id}_{label}_raw.pdb"
        shutil.move(pdb_file, str(clean_name))
        print(f"  Saved: {clean_name}")
    else:
        print(f"  WARNING: download failed for {pdb_id}")

# Clean up any empty PDB download subdirectories
for d in (STEP4_DIR / "receptors").iterdir():
    if d.is_dir():
        shutil.rmtree(d, ignore_errors=True)

print("\nReceptor files staged:")
for f in sorted((STEP4_DIR / "receptors").glob("*_raw.pdb")):
    print(f"  {f.name}")

## 2. PDB2PQR protonation

Runs PDB2PQR with PROPKA titration at pH 7.4 (physiological). This assigns protonation states to all titratable residues based on their predicted pKa values in the structural context, rather than accepting blanket defaults. The AMBER force field is used for charge/radius assignment, consistent with downstream Vina docking.

For ALDH3B2 specifically, Section 5b of Step 2 already identified the catalytic cysteine and titratable pocket-lining residues; this step now actually assigns their protonation states rather than just flagging them.

In [ ]:
import subprocess

PH = 7.4
FF = "AMBER"

raw_receptors = sorted((STEP4_DIR / "receptors").glob("*_raw.pdb"))
if not raw_receptors:
    raise FileNotFoundError("No raw receptor PDB files found. Run cell 1 first.")

for raw_pdb in raw_receptors:
    name = raw_pdb.stem.replace("_raw", "")
    pqr_output = STEP4_DIR / "receptors" / f"{name}_protonated.pqr"
    pdb_output = STEP4_DIR / "receptors" / f"{name}_protonated.pdb"

    print(f"\nProtonating {name} at pH {PH} with {FF} force field...")

    cmd = [
        "pdb2pqr30",
        f"--ff={FF}",
        f"--ffout={FF}",
        f"--with-ph={PH}",
        "--titration-state-method", "propka",
        "--drop-water",
        "--keep-chain",
        f"--pdb-output", str(pdb_output),
        str(raw_pdb),
        str(pqr_output)
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout[-500:] if result.stdout else "(no stdout)")
    if result.returncode != 0:
        print(f"  FAILED (exit code {result.returncode}):")
        print(result.stderr[-500:])
    else:
        if pdb_output.exists():
            print(f"  Protonated PDB: {pdb_output} ({pdb_output.stat().st_size} bytes)")
        if pqr_output.exists():
            print(f"  PQR file: {pqr_output}")

print("\nProtonation complete.")
for f in sorted((STEP4_DIR / "receptors").glob("*_protonated.*")):
    print(f"  {f.name}")

## 3. Receptor PDBQT conversion via Meeko

Converts the protonated PDB files to PDBQT format (the input format AutoDock Vina requires for receptors). Uses Meeko's `mk_prepare_receptor.py`, which also writes a JSON sidechain file needed for flexible-residue docking (not used in this pipeline's current scope, but good practice to generate alongside).

In [ ]:
receptor_pdbs = sorted((STEP4_DIR / "receptors").glob("*_protonated.pdb"))
if not receptor_pdbs:
    raise FileNotFoundError("No protonated receptor PDB files found. Run PDB2PQR cell first.")

for pdb_path in receptor_pdbs:
    name = pdb_path.stem.replace("_protonated", "")
    output_base = STEP4_DIR / "receptors" / name

    print(f"\nConverting {name} to PDBQT...")

    cmd = [
        "mk_prepare_receptor.py",
        "--read_pdb", str(pdb_path),
        "-o", str(output_base),
        "-p",   # write .pdbqt
        "-j",   # write .json (sidechain info)
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  mk_prepare_receptor.py failed:")
        print(result.stderr[-500:])
        # Fallback: try the older prepare_receptor4.py approach if meeko's new script fails
        print("  Attempting fallback via Meeko's Polymer Python API...")
        try:
            from meeko import Polymer
            with open(pdb_path) as f:
                pdb_string = f.read()
            polymer = Polymer.from_pdb_string(pdb_string)
            pdbqt_string = polymer.write_pdbqt_string()
            pdbqt_path = STEP4_DIR / "receptors" / f"{name}.pdbqt"
            pdbqt_path.write_text(pdbqt_string)
            print(f"  Fallback succeeded: {pdbqt_path}")
        except Exception as e:
            print(f"  Fallback also failed: {e}")
            print(f"  This receptor may need manual PDBQT preparation.")
    else:
        pdbqt_path = STEP4_DIR / "receptors" / f"{name}.pdbqt"
        if pdbqt_path.exists():
            print(f"  PDBQT: {pdbqt_path} ({pdbqt_path.stat().st_size} bytes)")
        json_path = STEP4_DIR / "receptors" / f"{name}.json"
        if json_path.exists():
            print(f"  JSON: {json_path}")

print("\nReceptor PDBQT files:")
for f in sorted((STEP4_DIR / "receptors").glob("*.pdbqt")):
    print(f"  {f.name}")

## 4. Ligand PDBQT preparation from the compound library

Converts the 328,869-compound SMILES library to docking-ready PDBQT format. Meeko can process ligands directly from SMILES strings (via RDKit internally), generating 3D coordinates, adding hydrogens, and writing PDBQT, without requiring intermediate SDF files on disk.

Given the library size, this is processed in chunks with Drive checkpointing, exactly the same pattern used in Steps 1B, 3, and the stereochemistry check, for the same reason: a session crash partway through should resume, not restart.

**This is the heaviest cell in this notebook.** At ~0.05-0.2 seconds per compound for SMILES-to-PDBQT conversion, 328,869 compounds could take 5-18 hours. The checkpoint logic means this can span multiple sessions safely.

In [ ]:
import pandas as pd
import time
import json as json_lib
from rdkit import Chem
from rdkit.Chem import AllChem

# Load compound library
library_path = STEP3_DIR / "library_final.csv"
if not library_path.exists():
    raise FileNotFoundError(f"Compound library not found at {library_path}. Run Step 3 first.")

library_df = pd.read_csv(library_path, low_memory=False)
smiles_col = 'canonical_smiles'
id_col = 'identifier' if 'identifier' in library_df.columns else 'id'

total = len(library_df)
print(f"Compound library loaded: {total} compounds")

LIGAND_PDBQT_DIR = STEP4_DIR / "ligands"
LIGAND_PDBQT_DIR.mkdir(exist_ok=True)
PROGRESS_PATH = STEP4_DIR / "logs" / "ligand_prep_progress.txt"
TIMING_LOG = STEP4_DIR / "logs" / "ligand_prep_timing.json"
CHUNK_SIZE = 5000

# Resume support
start_pos = int(PROGRESS_PATH.read_text()) if PROGRESS_PATH.exists() else 0
print(f"Resuming from compound {start_pos} / {total}")

chunk_timings = json_lib.loads(TIMING_LOG.read_text()) if TIMING_LOG.exists() else []

In [ ]:
from meeko import MoleculePreparation
from meeko import PDBQTWriterLegacy

# Process in chunks, writing individual PDBQT files per compound
# (Vina needs one PDBQT per ligand for batch docking)

n_success = 0
n_fail = 0

for start in range(start_pos, total, CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, total)
    chunk = library_df.iloc[start:end]

    t0 = time.time()
    chunk_success = 0
    chunk_fail = 0

    for _, row in chunk.iterrows():
        smi = row[smiles_col]
        compound_id = str(row[id_col])

        try:
            mol = Chem.MolFromSmiles(str(smi))
            if mol is None:
                chunk_fail += 1
                continue

            mol = Chem.AddHs(mol)
            result = AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
            if result == -1:
                chunk_fail += 1
                continue

            try:
                AllChem.MMFFOptimizeMolecule(mol, maxIters=200)
            except Exception:
                pass

            mk_prep = MoleculePreparation()
            molsetup_list = mk_prep(mol)
            if not molsetup_list:
                chunk_fail += 1
                continue

            pdbqt_string = PDBQTWriterLegacy.write_string(molsetup_list[0])

            out_path = LIGAND_PDBQT_DIR / f"{compound_id}.pdbqt"
            out_path.write_text(pdbqt_string)
            chunk_success += 1

        except Exception:
            chunk_fail += 1

    elapsed = time.time() - t0
    n_success += chunk_success
    n_fail += chunk_fail

    PROGRESS_PATH.write_text(str(end))

    per_mol = elapsed / max(len(chunk), 1)
    remaining = total - end
    est_min = (remaining * per_mol) / 60

    chunk_timings.append({
        'chunk_start': start, 'chunk_end': end, 'elapsed_s': round(elapsed, 1),
        'success': chunk_success, 'fail': chunk_fail,
        'sec_per_mol': round(per_mol, 4), 'timestamp': datetime.now().isoformat()
    })

    # Write timing log every 10 chunks to reduce Drive I/O
    if len(chunk_timings) % 10 == 0:
        TIMING_LOG.write_text(json_lib.dumps(chunk_timings, indent=2))

    print(f"  {end}/{total}: {chunk_success} ok, {chunk_fail} failed | "
          f"{elapsed:.1f}s ({per_mol:.3f}s/mol) | est. remaining: {est_min:.0f} min")

# Final timing log write
TIMING_LOG.write_text(json_lib.dumps(chunk_timings, indent=2))

print(f"\nLigand preparation complete.")
print(f"  Total success: {n_success}")
print(f"  Total failed: {n_fail}")
print(f"  PDBQT files in: {LIGAND_PDBQT_DIR}")

## 5. Save provenance and summary

In [ ]:
import json as json_lib
from datetime import datetime

n_receptor_pdbqt = len(list((STEP4_DIR / "receptors").glob("*.pdbqt")))
n_ligand_pdbqt = len(list(LIGAND_PDBQT_DIR.glob("*.pdbqt")))

provenance = {
    'timestamp': datetime.now().isoformat(),
    'receptors_prepared': n_receptor_pdbqt,
    'receptor_files': [f.name for f in sorted((STEP4_DIR / "receptors").glob("*.pdbqt"))],
    'protonation_method': 'PDB2PQR with PROPKA titration',
    'protonation_ph': 7.4,
    'force_field': 'AMBER',
    'receptor_pdbqt_tool': 'Meeko mk_prepare_receptor.py',
    'ligands_prepared': n_ligand_pdbqt,
    'ligand_pdbqt_tool': 'Meeko MoleculePreparation + PDBQTWriterLegacy',
    'ligand_source': str(STEP3_DIR / "library_final.csv"),
    'ligand_3d_method': 'RDKit ETKDGv3 embedding + MMFF94 optimization (maxIters=200)',
    'charmm_gui_note': 'Not used at this stage; CHARMM-GUI Solution Builder is relevant at Step 10 (MD system prep), not docking prep',
    'aldh3b2_source': 'AF3-predicted, validated in Step 2 (Section 5b)',
    'folr1_source': 'PDB 4LRH (folate-bound) and 5IZQ (antifolate AGF183-bound)',
    'folr1_role': 'Clinical benchmark comparator, not primary screening target',
}

with open(STEP4_DIR / "logs" / "step4_provenance.json", 'w') as f:
    json_lib.dump(provenance, f, indent=2)

print(f"Step 4 provenance saved.")
print(f"\n=== Step 4 complete ===")
print(f"  Receptors ready: {n_receptor_pdbqt}")
print(f"  Ligands ready: {n_ligand_pdbqt}")
print(f"\nStep 5 (docking validation) will load from: {STEP4_DIR}")
print(f"  Receptors: {STEP4_DIR / 'receptors'}")
print(f"  Ligands: {STEP4_DIR / 'ligands'}")